# Урок 26. Компьютерное зрение: как модель видит картинки

**Дата:** 18.03.2027 · **60 минут** · 8–9 класс  
**Модуль 5.** ИИ-приложения (уроки 26–29)  
**Опирается на:** уроки 19–21 (обучение, train / test, kNN, матрица ошибок), урок 13 (графики).

Распознавание лиц в телефоне, поиск по фото, автопилот, проверка брака на заводе — всё это **компьютерное зрение** (*computer vision, CV*). Сегодня разберёмся, как картинка превращается в числа, обучим модель распознавать рукописные цифры и увидим, как легко её обмануть.

## Цели урока

К концу занятия ты сможешь:

1. Объяснять, как компьютер хранит изображение.
2. Работать с массивами **numpy**: создавать, узнавать форму, менять форму.
3. Превращать картинку в набор признаков и обучать на них классификатор.
4. Находить, какие цифры модель путает, и смотреть на её ошибки.
5. Проверять модель на устойчивость к изменениям картинки.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–15 | Картинка — это числа; знакомство с numpy |
| 15–30 | Распознаём рукописные цифры |
| 30–40 | Ошибки и устойчивость модели |
| 40–55 | Практика: Teachable Machine или эксперименты в коде |
| 55–60 | Итог и домашнее задание |

## 1. Картинка — это таблица чисел

Экран состоит из маленьких точек — **пикселей**. Чёрно-белая картинка — это таблица, где в каждой клетке записана **яркость** пикселя: 0 — чёрный, большое число — белый.

Цветная картинка устроена так же, только у каждого пикселя **три** числа: сколько красного, зелёного и синего (**RGB**). Фото с телефона 4000 × 3000 пикселей — это 36 миллионов чисел.

Для работы с такими таблицами чисел в Python есть библиотека **numpy** (произносится «нам-пай»). Её основной объект — **массив**: похож на список, но может быть многомерным и умеет считать сразу над всеми элементами. Pandas и sklearn внутри построены на numpy. Принято подключать её как `np`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

smile = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 9, 0, 0, 0, 9, 0],
    [0, 9, 0, 0, 0, 9, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 9, 0, 0, 0, 9, 0],
    [0, 0, 9, 9, 9, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
])
print("Форма массива:", smile.shape)

plt.figure(figsize=(3, 3))
plt.imshow(smile, cmap="gray")
plt.title("Картинка 7 × 7 из чисел")
plt.show()

- `np.array(список списков)` — создать двумерный массив (таблицу чисел);
- `.shape` — форма: (строк, столбцов);
- `plt.imshow` — нарисовать массив как картинку; `cmap="gray"` — в оттенках серого.

**Попробуй сам:** поменяй несколько нулей на 9 — и нарисуй, например, нос.

### Операции над всем массивом сразу

С массивом можно делать арифметику без циклов — она применяется к **каждому** элементу. Например, «негатив» картинки: было светлое — стало тёмное.

In [ ]:
negative = 9 - smile
plt.figure(figsize=(3, 3))
plt.imshow(negative, cmap="gray")
plt.title("Негатив: 9 - smile")
plt.show()

### Из таблицы в строку

Модели из sklearn ждут на вход **строку признаков** для каждого объекта (как строка таблицы pandas). Поэтому картинку «вытягивают» в одну строку: 7 × 7 = 49 чисел — 49 признаков. Это делает `reshape`:

In [ ]:
row = smile.reshape(49)
print(row)
print("Форма:", row.shape)

Каждый пиксель стал **признаком**: «яркость в клетке (строка 2, столбец 6)». Модель не знает, что пиксели — соседи на картинке: для неё это просто 49 столбцов. Запомни это — скоро это объяснит ошибки модели.

## 2. Набор рукописных цифр

В sklearn встроен набор **digits**: почти 1800 картинок рукописных цифр 8 × 8 пикселей с правильными ответами. Яркость — от 0 до 16.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("Картинки:", digits.images.shape)
print("Признаки (картинки, вытянутые в строки):", digits.data.shape)
print("Ответы (первые 10):", digits.target[:10])

- `digits.images` — 1797 картинок 8 × 8 (трёхмерный массив: картинка, строка, столбец);
- `digits.data` — те же картинки, вытянутые в строки по 64 числа: **X**;
- `digits.target` — какая цифра нарисована: **y**.

Посмотрим на первые 10 картинок:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i in range(10):
    axes[i].imshow(digits.images[i], cmap="gray_r")
    axes[i].set_title(str(digits.target[i]))
    axes[i].axis("off")
plt.show()

`cmap="gray_r"` — обратная палитра: чёрные цифры на белом фоне. `axis("off")` прячет оси.

Картинки очень маленькие и грубые, но человек узнаёт цифры легко. Справится ли модель?

## 3. Обучаем распознавание цифр

Это классификация с **10 классами** (цифры 0–9). Всё как раньше: train / test, `fit`, `predict`, accuracy. Возьмём kNN: для новой картинки он найдёт самые похожие по пикселям картинки из train. Масштабировать признаки не нужно — все пиксели и так в одном масштабе (0–16).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X = digits.data
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)
print("Accuracy на test:", round(accuracy_score(y_test, pred), 3))

Почти 98% — модель распознаёт рукописные цифры, которых никогда не видела. Сравним с деревом решений:

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=10, random_state=0)
tree.fit(X_train, y_train)
print("Дерево, accuracy на test:", round(accuracy_score(y_test, tree.predict(X_test)), 3))

Дерево заметно хуже: вопросы вида «пиксель №37 ярче 5?» плохо подходят для картинок, где одна и та же цифра бывает нарисована чуть левее или правее.

## 4. Какие цифры модель путает

Для 10 классов матрица ошибок — таблица 10 × 10: строки — правда, столбцы — ответ модели. На диагонали — верные ответы.

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

Найди числа **вне** диагонали: какие цифры модель путала и с чем? Посмотрим на сами ошибки:

In [ ]:
wrong = []
for i in range(len(y_test)):
    if pred[i] != y_test[i]:
        wrong.append(i)
print("Ошибок:", len(wrong))

fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2.5))
for j in range(len(wrong)):
    i = wrong[j]
    axes[j].imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    axes[j].set_title(f"правда {y_test[i]}\nмодель {pred[i]}")
    axes[j].axis("off")
plt.show()

Многие ошибки понятны: некоторые цифры написаны так, что и человек засомневается. Разбирать ошибки глазами — обязательная часть работы инженера в компьютерном зрении.

## 5. Как легко обмануть модель

Модель видит не «цифру», а 64 отдельных числа. Что будет, если немного **сдвинуть** все картинки вправо? Человек сдвига даже не заметит.

Сначала превратим строки по 64 числа **обратно в картинки**: `X_test.reshape(len(X_test), 8, 8)` — «сделай столько картинок, сколько строк, каждую 8 × 8». Получится трёхмерный массив: номер картинки, строка, столбец. Эти три направления называют **осями** и нумеруют с нуля: ось 0 — картинка, ось 1 — строка, ось 2 — столбец.

`np.roll(массив, сдвиг, axis=2)` сдвигает пиксели вдоль оси 2, то есть **вправо** по столбцам (то, что «уехало» за правый край, появляется слева, — для наших картинок там почти всегда пусто). Потом снова вытягиваем картинки в строки по 64 числа, чтобы отдать модели.

Для **одной** картинки (двумерный массив: строка, столбец) столбцы — это ось 1, поэтому ниже при рисовании стоит `axis=1`.

In [ ]:
test_images = X_test.reshape(len(X_test), 8, 8)

for shift in [1, 2]:
    shifted = np.roll(test_images, shift, axis=2)
    shifted_rows = shifted.reshape(len(X_test), 64)
    acc = accuracy_score(y_test, knn.predict(shifted_rows))
    print(f"сдвиг на {shift} пикс.: accuracy {acc:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4, 2))
axes[0].imshow(test_images[0], cmap="gray_r")
axes[0].set_title("как было")
axes[1].imshow(np.roll(test_images[0], 2, axis=1), cmap="gray_r")
axes[1].set_title("сдвиг на 2")
for ax in axes:
    ax.axis("off")
plt.show()

Сдвиг на 2 пикселя — и модель почти разучилась узнавать цифры. Для неё «пиксель №37 яркий» и «пиксель №39 яркий» — совсем разные признаки. А ещё негатив:

In [ ]:
inverted = 16 - X_test
print("Негатив картинок: accuracy", round(accuracy_score(y_test, knn.predict(inverted)), 2))

**Вывод.** Модель хорошо работает только на картинках, **похожих на обучающие**: тот же размер, положение, цвет фона. Это главная слабость моделей компьютерного зрения:

- сфотографировали при другом свете — модель ошибается;
- объект в другом углу кадра или под другим углом — ошибается;
- модель может «выучить» не объект, а **фон**: если все кошки в train сняты на диване, модель решит, что «диван = кошка».

Как с этим борются:

1. **Больше разнообразных данных** — разный свет, фон, ракурс.
2. **Аугментация** — искусственно размножают данные: сдвигают, поворачивают, меняют яркость картинок из train.
3. **Нейросети для изображений** (*свёрточные нейронные сети*) — модели, которые учитывают, что пиксели — соседи. Они лежат в основе современного CV, но обучаются на миллионах картинок.

## 6. Этика компьютерного зрения

- **Лица — персональные данные.** Фотографировать и распознавать людей можно только с их согласия.
- **Смещение.** Если в данных мало людей какой-то группы, система будет работать на них хуже (урок 24).
- **Цена ошибки.** Ошибка в фильтре фото — мелочь; ошибка в медицинском снимке или в системе наблюдения — серьёзно. Решения о людях должен проверять человек.

## 7. Практика

Выбери **вариант A** (без кода, в браузере) или **вариант B** (эксперименты в коде). Если успеваешь — сделай оба.

### Вариант A. Свой классификатор в Teachable Machine (20 минут)

[Teachable Machine](https://teachablemachine.withgoogle.com) — бесплатный сайт Google, где можно обучить модель распознавания изображений с веб-камеры прямо в браузере.

1. Открой сайт → **Get Started** → **Image Project** → **Standard image model**.
2. Создай 3 класса, например жесты: «кулак», «ладонь», «ок».
3. Для каждого класса запиши 30–50 кадров с веб-камеры: меняй положение руки и расстояние.
4. Нажми **Train Model**.
5. **Проверь модель как инженер** — заполни дневник ниже.

**Важно:** снимай только руки, а не лица; не загружай фото других людей.

**Дневник эксперимента:**

| Проверка | Что получилось |
|----------|----------------|
| Тот же жест, та же рука, тот же фон | |
| Другой рукой | |
| При другом освещении | |
| На другом фоне | |
| Предмет, которого не было в обучении (например, кружка) | |
| Худший случай: где модель ошибается сильнее всего | |
| Почему, по-твоему, она там ошибается? | |
| Что изменить в данных, чтобы стало лучше? | |

### Вариант B. Эксперименты в коде (20 минут)

**B1. Сколько нужно данных?** Обучи kNN на первых 20, 50, 100 и 300 картинках из train и проверь на всём test. Как качество зависит от количества примеров?

In [ ]:
# B1

<details>
<summary>Решение B1</summary>

```python
for n in [20, 50, 100, 300]:
    model = KNeighborsClassifier(n_neighbors=3)
    model.fit(X_train[:n], y_train[:n])
    print(f"{n:3} картинок: accuracy {accuracy_score(y_test, model.predict(X_test)):.2f}")
```
С 20 примерами (по два на цифру) модель ошибается чаще, чем угадывает; с 300 — уже около 95%. Для компьютерного зрения данные — главное.
</details>

**B2. Аугментация.** Добавь в train копии картинок, сдвинутые на 1 пиксель вправо и влево. Обучи kNN на расширенном train и снова проверь на test, сдвинутом на 1 пиксель. Помогло?

Подсказка: `np.concatenate([a, b, c])` склеивает массивы друг под другом.

In [ ]:
# B2

<details>
<summary>Решение B2</summary>

```python
train_images = X_train.reshape(len(X_train), 8, 8)
right = np.roll(train_images, 1, axis=2).reshape(len(X_train), 64)
left = np.roll(train_images, -1, axis=2).reshape(len(X_train), 64)

X_aug = np.concatenate([X_train, right, left])
y_aug = np.concatenate([y_train, y_train, y_train])

knn_aug = KNeighborsClassifier(n_neighbors=3)
knn_aug.fit(X_aug, y_aug)

shifted_test = np.roll(test_images, 1, axis=2).reshape(len(X_test), 64)
print("Обычная модель на сдвинутом test:", round(accuracy_score(y_test, knn.predict(shifted_test)), 2))
print("Модель с аугментацией:", round(accuracy_score(y_test, knn_aug.predict(shifted_test)), 2))
```
Аугментация научила модель, что сдвинутая цифра — та же цифра.
</details>

## Работа с ИИ-помощником

- «Объясни, почему сдвиг картинки на 2 пикселя так сильно портит kNN.»
- «Что такое аугментация данных? Приведи примеры для фото.»
- «Чем свёрточная нейросеть отличается от kNN на пикселях? Объясни для 9 класса.»

## Мини-глоссарий урока 26

| Термин | Смысл |
|--------|-------|
| **Компьютерное зрение (CV)** | ИИ для работы с изображениями и видео |
| **Пиксель** | точка изображения; хранится как число (или три числа RGB) |
| **numpy, массив** | библиотека и объект для таблиц чисел любой размерности |
| **`shape` / `reshape`** | форма массива / изменить форму |
| **Аугментация** | искусственное расширение данных: сдвиги, повороты, яркость |
| **Устойчивость модели** | способность работать при небольших изменениях входа |

## Итог урока

1. Картинка для компьютера — массив чисел; пиксели становятся признаками.
2. Модель хорошо распознаёт картинки, похожие на обучающие, и легко ошибается при сдвиге, другом свете или фоне.
3. Против этого помогают разнообразные данные, аугментация и специальные нейросети.

## Домашнее задание (20 минут)

Найди пример системы компьютерного зрения в жизни (разблокировка телефона, распознавание номеров, сортировка фото, медицина). Ответь:

1. Что на входе и что на выходе системы? Какая это задача — классификация или другая?
2. При каких условиях она может ошибаться (свет, ракурс, фон, люди разной внешности)?
3. Какие этические вопросы возникают (приватность, согласие, цена ошибки)?

**ДЗ:** …

## Самопроверка

1. Как хранится чёрно-белая и цветная картинка?
2. Зачем картинку «вытягивают» в строку перед обучением?
3. Почему kNN на пикселях ломается от сдвига картинки?
4. Что такое аугментация и как она помогает?
5. Какие данные в компьютерном зрении считаются персональными?

**Следующий урок (25.03.2027):** текст и чат-боты — как модели работают со словами.